# Cuffless AI — Naive Baseline

This notebook creates a simple baseline for blood pressure prediction. The baseline always predicts the average SBP and DBP from the training data. Future models should perform better than this baseline.

## 1. Imports

In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 2. Locate the data files

In [ ]:
data_files = [
    Path("../data/Part_1.mat"),
    Path("../data/Part_2.mat"),
    Path("../data/Part_3.mat"),
    Path("../data/Part_4.mat"),
]

for filename in data_files:
    print(filename.name, "exists:", filename.exists())

if not all(filename.exists() for filename in data_files):
    raise FileNotFoundError("Put all four Part_*.mat files inside the data/ folder.")

## 3. Create five-second SBP and DBP labels

In [ ]:
FS = 125
WINDOW_SECONDS = 5
WINDOW_SAMPLES = FS * WINDOW_SECONDS

sbp_labels = []
dbp_labels = []
recording_groups = []

for file_number, filename in enumerate(data_files):
    dataset_name = filename.stem

    with h5py.File(filename, "r") as f:
        recording_refs = f[dataset_name]

        for record_index in range(recording_refs.shape[0]):
            record = np.asarray(f[recording_refs[record_index, 0]])

            if record.shape[1] != 3:
                record = record.T

            num_windows = len(record) // WINDOW_SAMPLES
            abp = record[:num_windows * WINDOW_SAMPLES, 1]
            abp_windows = abp.reshape(num_windows, WINDOW_SAMPLES)

            sbp_labels.extend(abp_windows.max(axis=1))
            dbp_labels.extend(abp_windows.min(axis=1))

            group_id = file_number * 3000 + record_index
            recording_groups.extend([group_id] * num_windows)

    print(f"Finished {filename.name}")

sbp_labels = np.asarray(sbp_labels)
dbp_labels = np.asarray(dbp_labels)
recording_groups = np.asarray(recording_groups)

print("Total windows:", len(sbp_labels))

## 4. Create training and test sets

All windows from one recording stay together. This prevents the same recording from appearing in both training and testing.

In [ ]:
X = np.zeros((len(sbp_labels), 1))

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_index, test_index = next(
    splitter.split(X, groups=recording_groups)
)

X_train = X[train_index]
X_test = X[test_index]
sbp_train = sbp_labels[train_index]
sbp_test = sbp_labels[test_index]
dbp_train = dbp_labels[train_index]
dbp_test = dbp_labels[test_index]

print("Training windows:", len(train_index))
print("Test windows:", len(test_index))

## 5. Train the naive baselines

In [ ]:
sbp_baseline = DummyRegressor(strategy="mean")
dbp_baseline = DummyRegressor(strategy="mean")

sbp_baseline.fit(X_train, sbp_train)
dbp_baseline.fit(X_train, dbp_train)

sbp_predictions = sbp_baseline.predict(X_test)
dbp_predictions = dbp_baseline.predict(X_test)

print(f"Baseline SBP prediction: {sbp_predictions[0]:.2f} mmHg")
print(f"Baseline DBP prediction: {dbp_predictions[0]:.2f} mmHg")

## 6. Evaluate the baseline

In [ ]:
results = pd.DataFrame({
    "Target": ["SBP", "DBP"],
    "MAE": [
        mean_absolute_error(sbp_test, sbp_predictions),
        mean_absolute_error(dbp_test, dbp_predictions),
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(sbp_test, sbp_predictions)),
        np.sqrt(mean_squared_error(dbp_test, dbp_predictions)),
    ],
})

print(results.round(2).to_string(index=False))

## What the results mean

The baseline does not use the PPG signal. It predicts the same average SBP and DBP for every test window. A useful machine-learning model should produce lower MAE and RMSE values than this baseline.

Verified results with the current four files:

- SBP: MAE 18.67 mmHg, RMSE 22.93 mmHg
- DBP: MAE 8.62 mmHg, RMSE 11.17 mmHg